# Phase 3 Synthetic Spend & Margin Layers

**Project:** channel-profitability-audit

**Purpose:** Construct the two synthetic layers needed to test whether ROAS ranking
matches true profitability ranking: (1) channel-level ad spend, (2) apparel gross margin.
Real GA channel volume × period-correct (2016-17) benchmark rates, per
`docs/01-data-decision.md` and `docs/decision-log.md`.

**Scope:** Paid Search only gets a real spend line. Referral, Organic Search, and
Direct are $0 spend by definition (organic channels) so do not apply CAC/CPC benchmarks
to them.

**Reference:** See `docs/decision-log.md` and "Phase 3 — Synthetic layer sourcing" and
"Phase 3 - Margin base case finalized" entries for full sourcing rationale, rejected
alternatives, and unresolved limitations (CPA-as-CAC ambiguity, COGS-scope mismatch).

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

## 1. Benchmark constants

Hardcoded, named, and commented with source not inline magic numbers.
Every value here should trace back to a `docs/decision-log.md` entry.

**Open limitations carried into this layer:**
- Paid Search CPA is treated as a purchase-equivalent CAC proxy. Source does not
  disambiguate purchase vs. lead-generation conversions for the e-commerce vertical.
- Margin base case (45.93%) is a midpoint between two companies with different COGS
  scope (shipping in/out), not a normalized industry estimate.

In [2]:
# Paid Search benchmark (WordStream Google Ads, e-commerce vertical) 
# Two periods, live page + Wayback archive, see decision-log.md
PAID_SEARCH_CPC_BASE = 1.02
PAID_SEARCH_CPA_BASE = 45.67       # CPA-as-CAC assumption, see limitation note
PAID_SEARCH_CPC_RANGE = (0.88, 1.16)
PAID_SEARCH_CPA_RANGE = (45.27, 46.07)

# Display benchmark (same WordStream source as Paid Search, Display column)
DISPLAY_CPC_BASE = 0.37
DISPLAY_CPA_BASE = 48.01           # same CPA-as-CAC assumption; wide range below reflects real instability
DISPLAY_CPC_RANGE = (0.29, 0.45)
DISPLAY_CPA_RANGE = (30.21, 65.80)

# Social benchmark (WordStream Facebook Ads, Apparel vertical, Nov 2016-Jan 2017)
# Source: https://www.wordstream.com/blog/ws/2017/02/28/facebook-advertising-benchmarks
# Single period only - no range available, no second-year anchor found
SOCIAL_CPC_BASE = 0.45
SOCIAL_CPA_BASE = 10.98            # same CPA-as-CAC assumption as Search/Display

# Affiliates: no synthetic spend
# No period-correct commission-rate benchmark found (2016-17). Given n=9
# converting sessions (thinnest channel, Phase 2 insufficient-volume flag),
# forcing a modern-vintage estimate would compound two reliability problems.
# Carry real GA volume/revenue only - no spend, no CAC, no payback for this channel.
AFFILIATES_SYNTHETIC_SPEND = None  # explicit flag


# Apparel DTC gross margin (Revolve + Stitch Fix S-1 filings, FY2016/FY2017)
MARGIN_BASE_CASE = 0.4593
MARGIN_RANGE = (0.4426, 0.4847)